# Original ViT-Base 30% KTA / 12% ATS experiment

Source: `i_100_drop_using_kendell/kendall-Copy1.ipynb`. The original saved output ends at **81.50% accuracy on 10,000 images**. Compared with the paper FX16 baseline of **83.35%**, this is a **1.85 percentage-point drop**. The paper reports **49.91% savings / 8.80 GOPs**; this notebook does not measure those costs.

The full historical numerical implementation is retained. Machine-specific paths, CUDA device, output location, and class-folder extraction have been made portable. Only the final 11 progress lines are retained as **historical output**, not evidence of a new run. The original loop appends to its output log; choose a fresh output directory for each run.

Set data and labels in the next cell. This original code retains its historical execution semantics, including the absence of an explicit `eval()` call; use the companion focused notebook and shared evaluator for the maintained, tested workflow. Original subset/checkpoint details remain necessary for exact reproduction.

In [ ]:
patch_drop_blocks=[3,4,5,6,7,8]
drop=0.300
attention_pruning= 0.12

# Set these paths before running the historical experiment.
from pathlib import Path
DATA_DIR = Path('/path/to/imagenet/val')
CLASS_INDEX = Path('/path/to/imagenet_class_index.json')
OUTPUT_DIR = Path('runs/original_vit_base_30_12')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import matplotlib.pyplot as plt
from time import time
import numpy as np
import os
import cv2
import json
import torch
from timm.models.vision_transformer import VisionTransformer, PatchEmbed
import torch
import torch.nn as nn
import numpy as np
import os
import re
import timm

In [ ]:
drop_rate= 1.00 -drop
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
img_folder = str(DATA_DIR)
txt_path = f"vit_base_similarity_dropblock{patch_drop_blocks}_keep_rate{drop_rate:.2f}_attnprune{attention_pruning:.2f}.txt"
json_file = str(CLASS_INDEX)
# line_r=0
 
txt_path = str(OUTPUT_DIR / Path(txt_path).name)
assert DATA_DIR.is_dir() and CLASS_INDEX.is_file(), "Configure DATA_DIR and CLASS_INDEX first"


In [ ]:
# pretrained_vit = vit_large_patch16_224(pretrained=True).to(device)

pretrained_vit = timm.create_model('vit_base_patch16_224', pretrained=True).to(device)

In [ ]:
def plot(image):
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.show()

In [ ]:
def copy_image(img):
    """
    Create a copy of the input image without using any libraries.

    Args:
    - img: Input image (list of lists representing pixel values)

    Returns:
    - img_copy: A copy of the input image
    """
    # Determine the dimensions of the image
    height = len(img)
    width = len(img[0])

    # Create an empty list to store the copied image
    img_copy = []

    # Iterate over each row of the image
    for i in range(height):
        # Create a copy of the current row
        row_copy = img[i][:]  # or row_copy = list(img[i]) for deep copy
        # Append the copied row to the copied image
        img_copy.append(row_copy)

    return img_copy

In [ ]:
def bgr_to_rgb(img):
    """
    Convert an image from BGR to RGB color space.

    Args:
    - img: Input image in BGR format (numpy array)

    Returns:
    - img_rgb: Image converted to RGB color space
    """
    # Reverse the order of color channels
    img_rgb = img[:, :, ::-1]

    return img_rgb

In [ ]:
def resize_nearest_neighbor(image, size):
    original_height = len(image)
    original_width = len(image[0])
    new_width, new_height = size

    if isinstance(image[0][0], list) or isinstance(image[0][0], np.ndarray):
        # For color images
        resized_image = np.zeros((new_height, new_width, 3), dtype=np.uint8)
        for i in range(new_height):
            for j in range(new_width):
                # Calculate the nearest pixel in the original image
                x = int(j * original_width / new_width)
                y = int(i * original_height / new_height)
                resized_image[i, j] = image[y][x]
    else:
        # For grayscale images
        resized_image = np.zeros((new_height, new_width), dtype=np.uint8)
        for i in range(new_height):
            for j in range(new_width):
                # Calculate the nearest pixel in the original image
                x = int(j * original_width / new_width)
                y = int(i * original_height / new_height)
                resized_image[i, j] = image[y][x]
    
    return np.array(resized_image)

In [ ]:
def save_average_percentage(file_path, average_percent, accuracy, processed_image, w):
    with open(file_path, 'a') as file:
        file.write(f"Epoch: Average percentage_patch_drop: {average_percent}, Accuracy: {accuracy}, processed_image: {processed_image}, scan: {w}\n")

In [ ]:
def load_mapping_file(json_file):
    with open(json_file, 'r') as f:
        mapping_data = json.load(f)
    return mapping_data

In [ ]:
import torchvision.transforms as transforms
def transform_image(image, device):

    image=transforms.ToTensor()(image)
    image= image.unsqueeze(0)
    image=image.to(device)    
    

    
    return image

In [ ]:
mapping_data = load_mapping_file(json_file)

def get_key_by_value(predicted_index, class_name):
    
    for key, values in mapping_data.items():
        key = int(key)
        
        if class_name in values:
            # print(key, predicted_index, "key and predicted_index")
            # print(f"Found class_name '{class_name}' in values: {values}")
            if int(key) == predicted_index:
                return True
    return False

In [ ]:
patch_drop_percentage = []
def average_patch_drop(mask):
    count_ones = mask.count(1)
    count_ones=count_ones-1
    y = (((196 - count_ones ) / 196) * 100)
    patch_drop_percentage.append(y)
    

In [ ]:
def compute_inv_covariance(x):
    # x: (batch_size, num_tokens, embedding_dim)
    # Flatten batch and token dimensions
    B, N, D = x.shape
    x_flat = x.reshape(-1, D)  # (B*N, D)
    
    # Compute mean and center
    mean = x_flat.mean(dim=0, keepdim=True)
    centered = x_flat - mean
    
    # Compute covariance: (D, D)
    cov = centered.T @ centered / (x_flat.shape[0] - 1)
    
    # Regularization for numerical stability
    eps = 1e-6
    cov += eps * torch.eye(D, device=x.device)
    
    # Invert covariance matrix
    inv_cov = torch.inverse(cov)
    return inv_cov  # (D, D)

def mahalanobis_distance(x, y, inv_cov):
    # x: (B, N, D), y: (B, D), inv_cov: (D, D)
    diff = x - y.unsqueeze(1)  # (B, N, D)
    left = torch.matmul(diff, inv_cov)  # (B, N, D)
    mahal = (left * diff).sum(dim=-1).sqrt()  # (B, N)
    return mahal

In [ ]:
import torch

def pearson_corr(cls_token, tokens):
    """
    Computes Kendall's Tau rank correlation.

    WARNING: This is a naive, conceptual implementation with O(D^2) complexity
    per token pair, making it extremely slow for typical embedding dimensions.
    It is for educational purposes and not suitable for production use.
    """
    B, N, D = tokens.shape
    
    # Expand cls_token to compare with each patch token
    cls_expanded = cls_token.unsqueeze(1) # (B, 1, D)
    
    # Create pairwise comparison matrices
    cls_diff = cls_expanded.unsqueeze(3) - cls_expanded.unsqueeze(2) # (B, 1, D, D)
    tokens_diff = tokens.unsqueeze(3) - tokens.unsqueeze(2) # (B, N, D, D)
    
    # Determine concordant and discordant pairs
    # sign() will be 1 if positive, -1 if negative, 0 if zero
    concordant_matrix = torch.sign(cls_diff) * torch.sign(tokens_diff)
    
    # Sum up concordant (1) and discordant (-1) pairs, ignoring diagonal and duplicates
    # We use triu (upper triangle) to count each pair only once.
    num_concordant_discordant = torch.triu(concordant_matrix, diagonal=1).sum(dim=(2, 3))
    
    # Total number of pairs is D * (D - 1) / 2
    total_pairs = D * (D - 1) / 2
    
    tau = num_concordant_discordant / total_pairs
    return tau

In [ ]:
token_counts_per_image = []

In [ ]:
import torch
import torch.nn as nn
import timm
from timm.models.vision_transformer import VisionTransformer, Block
import numpy as np
import torch.nn.functional as F
class CustomAttention(nn.Module):
    def __init__(self, attn, num_heads):
        super(CustomAttention, self).__init__()
        self.num_heads = num_heads
        self.scale = attn.scale
        self.qkv = attn.qkv
        self.attn_drop = attn.attn_drop
        self.proj = attn.proj
        self.proj_drop = attn.proj_drop

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        mask_attention = None
        avg_attn = attn.mean(dim=1)
        cls_token_attn = avg_attn[:, 0, :] 
        threshold = torch.quantile(cls_token_attn, attention_pruning , dim=-1, keepdim=True)
        mask_attention = cls_token_attn >= threshold
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x, mask_attention


class PatchDropout(nn.Module):
    def __init__(self):
        super().__init__()
        # self.drop_rate = drop_rate  # Drop 20% of the tokens

    def forward(self, x, mask):
        batch_size, seq_len, embedding_dim = x.shape
        assert seq_len == 197, f"Expected sequence length of 197, but got {seq_len}"
        
        # Extract CLS token
        cls_token = x[:, 0, :]  # (B, D)
        tokens = x[:, 1:, :]    # (B, 196, D)
    
        # Recompute inverse covariance matrix from all tokens (excluding CLS)
        # inv_cov = compute_inv_covariance(tokens)  # (D, D)
        distances = pearson_corr(cls_token, tokens)
        # Compute Mahalanobis distance between each token and CLS
        # distances = mahalanobis_distance(tokens, cls_token, inv_cov)  # (B, 196)

    
        # Keep bottom 80% farthest tokens (drop closest 20%)
        k = int((seq_len - 1) * drop_rate)  # 20% of 196 tokens
        threshold_values, _ = torch.kthvalue(distances, k + 1, dim=1, keepdim=True)  # (B, 1)
    
        keep_mask = distances <= threshold_values  # (B, 196)
        keep_mask = torch.cat([torch.ones(batch_size, 1, dtype=torch.bool, device=x.device), keep_mask], dim=1)  # (B, 197)
    
        # Apply mask to x
        x_filtered = x[keep_mask].view(batch_size, -1, embedding_dim)
        # print(x_filtered.shape)
        # token_counts_per_image.append(x_filtered.shape[1])
        
        return x_filtered

class CustomBlock(nn.Module):
    def __init__(self, block):
        super(CustomBlock, self).__init__()
        self.norm1 = block.norm1
        self.attn = CustomAttention(block.attn, block.attn.num_heads)
        self.ls1 = block.ls1
        self.drop_path1 = block.drop_path1
        self.norm2 = block.norm2
        self.mlp = block.mlp
        self.ls2 = block.ls2
        self.drop_path2 = block.drop_path2

    def forward(self, x, apply_patch_drop=False):

        
        x_residual = self.norm1(x)
        x_residual, mask_attention = self.attn(x_residual)
        x = x + self.drop_path1(self.ls1(x_residual))
        x = x + self.drop_path2(self.ls2(self.mlp(self.norm2(x))))
        if apply_patch_drop:
            
            cls_token = x[:, 0].unsqueeze(1)  # Keep the class token
            other_tokens = x[:, 1:]
            selected_tokens = other_tokens[mask_attention[:, 1:]] 
            selected_tokens = selected_tokens.unsqueeze(0)
            x = torch.cat((cls_token, selected_tokens), dim=1)
            
        return x

class CustomVisionTransformer(VisionTransformer):
    def __init__(self, patch_drop_blocks=None, **kwargs):
        super(CustomVisionTransformer, self).__init__(**kwargs)
        self.patch_drop_blocks = patch_drop_blocks if patch_drop_blocks is not None else []
        self.patch_drop = PatchDropout()
        self.blocks = nn.Sequential(*[
            CustomBlock(block) if i in self.patch_drop_blocks else block
            for i, block in enumerate(self.blocks)
        ])

    def forward(self, x, mask=None):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)  # class token
        x = torch.cat((cls_tokens, x), dim=1)
        x = self.pos_drop(x + self.pos_embed)
        if mask is not None:
            x = self.patch_drop(x, mask)

        for i, block in enumerate(self.blocks):
            print(x.shape)
            if isinstance(block, CustomBlock):
                x = block(x, apply_patch_drop=True)
                # token_counts_per_image.append(x.shape[1])
            else:
                x = block(x)
                # token_counts_per_image.append(x.shape[1])
        x = self.norm(x)
        x = self.head(x[:, 0])
        return x

# Example usage
my_array = np.random.randint(1, 2, size=197)  # Updated to have values 0 or 1
custom_vit = CustomVisionTransformer(
    img_size=224, patch_size=16, embed_dim=768, depth=12, num_heads=12, mlp_ratio=4, qkv_bias=True,
    norm_layer=nn.LayerNorm, num_classes=1000, patch_drop_blocks= patch_drop_blocks
).to(device)


# Load pretrained weights
# pretrained_vit = timm.create_model('vit_base_patch16_224', pretrained=True)
custom_vit.load_state_dict(pretrained_vit.state_dict(), strict=True)

# Forward pass with input tensor and masks
input_tensor = torch.randn(1, 3, 224, 224).to(device)
output = custom_vit(input_tensor, mask=my_array)
print(output.shape)  # Should be [1, 1000]


In [ ]:
from qtorch.quant import Quantizer, quantizer
from qtorch import FixedPoint
# Define the fixed-point quantizer
weight_quantizer = quantizer(forward_number=FixedPoint(16, 8), forward_rounding="nearest")
act_quantizer = quantizer(forward_number=FixedPoint(16, 8), forward_rounding="nearest")

In [ ]:
# import torch
# import torch.nn as nn

# # Define the QuantizedVisionTransformer class
# class QuantizedVisionTransformer(nn.Module):
#     def __init__(self, model, weight_quantizer, act_quantizer):
#         super(QuantizedVisionTransformer, self).__init__()
#         self.model = model
#         self.weight_quantizer = weight_quantizer
#         self.act_quantizer = act_quantizer
#         self.apply(self._quantize_weights)

#     def _quantize_weights(self, module):
#         if isinstance(module, nn.Linear):
#             module.weight.data = self.weight_quantizer(module.weight.data)
#             if module.bias is not None:
#                 module.bias.data = self.weight_quantizer(module.bias.data)
#         elif isinstance(module, nn.Conv2d):
#             module.weight.data = self.weight_quantizer(module.weight.data)
#             if module.bias is not None:
#                 module.bias.data = self.weight_quantizer(module.bias.data)

#     def forward(self, x, mask):
#         x = self.act_quantizer(x)
#         # for name, module in self.model.named_children():
#         #     x = module(x, mask) if 'block' in name else module(x)
#         #     if isinstance(module, (nn.Linear, nn.Conv2d)):
#         #         x = self.act_quantizer(x)
#         return self.model(x, mask)

# # Example usage:
# # Assume `vision_transformer_model` is your pretrained model
# # and `weight_quantizer` and `act_quantizer` are your quantization functions
# quantized_model = QuantizedVisionTransformer(custom_vit, weight_quantizer, act_quantizer)



In [ ]:
import torch
import torch.nn as nn
from qtorch.quant import quantizer
from qtorch import FixedPoint

# =====================================================
# INT8 Quantizers Configuration
# =====================================================
weight_quantizer = quantizer(
    forward_number=FixedPoint(16, 8),   # INT8
    forward_rounding="nearest"
)

act_quantizer = quantizer(
    forward_number=FixedPoint(16, 8),   # INT8
    forward_rounding="nearest"
)

# =====================================================
# Complete Quantized Vision Transformer
# =====================================================
class QuantizedVisionTransformer(nn.Module):
    """
    Comprehensive INT8 quantization wrapper for Vision Transformer.
    
    Quantizes:
    - All weights (Linear, Conv2d, Embedding, LayerNorm params)
    - All biases
    - All activations (Linear, Conv2d, Embedding, LayerNorm, GELU, Attention outputs)
    - Input and output
    """
    
    def __init__(self, model, weight_quantizer, act_quantizer):
        super().__init__()
        self.model = model
        self.wq = weight_quantizer
        self.aq = act_quantizer
        
        # Quantize all weights ONCE during initialization
        self._quantize_all_weights()
        
        # Register hooks for activation quantization
        self._register_activation_hooks()
    
    # =====================================================
    # Weight Quantization (All Layers)
    # =====================================================
    def _quantize_all_weights(self):
        """
        Quantize weights and biases across ALL layer types:
        - Linear layers
        - Conv2d layers
        - Embedding layers
        - LayerNorm parameters (scale & shift)
        """
        for module in self.model.modules():
            # Linear layers (includes all projection layers in attention & MLP)
            if isinstance(module, nn.Linear):
                module.weight.data = self.wq(module.weight.data)
                if module.bias is not None:
                    module.bias.data = self.wq(module.bias.data)
            
            # Conv2d layers (patch embedding in ViT)
            elif isinstance(module, nn.Conv2d):
                module.weight.data = self.wq(module.weight.data)
                if module.bias is not None:
                    module.bias.data = self.wq(module.bias.data)
            
            # Embedding layers (token, position, class embeddings)
            elif isinstance(module, nn.Embedding):
                module.weight.data = self.wq(module.weight.data)
            
            # LayerNorm parameters (scale and shift)
            elif isinstance(module, nn.LayerNorm):
                if module.weight is not None:
                    module.weight.data = self.wq(module.weight.data)
                if module.bias is not None:
                    module.bias.data = self.wq(module.bias.data)
    
    # =====================================================
    # Activation Quantization Hooks
    # =====================================================
    def _activation_hook(self, module, input, output):
        """
        Hook function to quantize activations after forward pass.
        Handles both single tensor and tuple outputs.
        """
        if isinstance(output, torch.Tensor):
            return self.aq(output)
        elif isinstance(output, tuple):
            # For layers that return multiple outputs (e.g., some attention implementations)
            return tuple(self.aq(o) if isinstance(o, torch.Tensor) else o for o in output)
        return output
    
    # =====================================================
    # Hook Registration (Comprehensive)
    # =====================================================
    def _register_activation_hooks(self):
        """
        Register hooks on ALL layers that produce activations:
        - Linear (QKV, output projections, MLP layers)
        - Conv2d (patch embedding)
        - Embedding (token embeddings)
        - LayerNorm (normalization outputs)
        - GELU (activation function)
        - Softmax (attention weights) - if accessible
        """
        hook_layers = (
            nn.Linear,
            nn.Conv2d,
            nn.Embedding,
            nn.LayerNorm,
            nn.GELU,
            nn.ReLU,
            nn.Softmax
        )
        
        for module in self.model.modules():
            if isinstance(module, hook_layers):
                module.register_forward_hook(self._activation_hook)
    
    # =====================================================
    # Forward Pass
    # =====================================================
    def forward(self, x, mask=None):
        """
        Forward pass with comprehensive quantization:
        1. Quantize input
        2. Process through model (weights already quantized, activations quantized via hooks)
        3. Quantize output
        """
        # Quantize input
        x = self.aq(x)
        
        # Forward pass through model
        # Weights are already quantized (done once in __init__)
        # Activations are quantized via registered hooks
        if mask is not None:
            x = self.model(x, mask)
        else:
            x = self.model(x)
        
        # Quantize final output
        x = self.aq(x)
        
        return x
    
    # =====================================================
    # Utility Methods
    # =====================================================
    def get_quantization_stats(self):
        """
        Print statistics about quantized layers.
        """
        layer_count = {
            'Linear': 0,
            'Conv2d': 0,
            'Embedding': 0,
            'LayerNorm': 0,
            'Activation': 0
        }
        
        for module in self.model.modules():
            if isinstance(module, nn.Linear):
                layer_count['Linear'] += 1
            elif isinstance(module, nn.Conv2d):
                layer_count['Conv2d'] += 1
            elif isinstance(module, nn.Embedding):
                layer_count['Embedding'] += 1
            elif isinstance(module, nn.LayerNorm):
                layer_count['LayerNorm'] += 1
        
        # Activation hooks count
        layer_count['Activation'] = sum(
            1 for m in self.model.modules() 
            if isinstance(m, (nn.Linear, nn.Conv2d, nn.Embedding, nn.LayerNorm, nn.GELU))
        )
        
        print("\n" + "="*60)
        print("QUANTIZATION STATISTICS")
        print("="*60)
        print(f"Linear Layers (Quantized):      {layer_count['Linear']}")
        print(f"Conv2d Layers (Quantized):      {layer_count['Conv2d']}")
        print(f"Embedding Layers (Quantized):   {layer_count['Embedding']}")
        print(f"LayerNorm Layers (Quantized):   {layer_count['LayerNorm']}")
        print(f"Total Activation Hooks:         {layer_count['Activation']}")
        print("="*60)
        print(f"Weight Quantization: INT8 (FixedPoint 16,8)")
        print(f"Activation Quantization: INT8 (FixedPoint 16,8)")
        print("="*60 + "\n")
        
        return layer_count
    
    def disable_quantization(self):
        """
        Remove all activation hooks (keeps weights quantized).
        """
        for module in self.model.modules():
            module._forward_hooks.clear()
    
    def enable_quantization(self):
        """
        Re-register activation hooks.
        """
        self._register_activation_hooks()


# =====================================================

quantized_model = QuantizedVisionTransformer(
    custom_vit,
    weight_quantizer,
    act_quantizer
)

quantized_model.get_quantization_stats()
    
# Forward pass
dummy_input = torch.randn(1, 3, 224, 224).to(device)
output = quantized_model(dummy_input, mask=my_array)
print(f"Output shape: {output.shape}")

print("Quantized Vision Transformer Module Loaded Successfully!")
print("\nKey Improvements:")
print("✓ Weights quantized ONCE in __init__ (not every forward pass)")
print("✓ All layer types covered: Linear, Conv2d, Embedding, LayerNorm")
print("✓ Comprehensive activation quantization via hooks")
print("✓ Handles both single tensor and tuple outputs")
print("✓ Includes utility methods for statistics and control")

In [ ]:
def check_quantization(model):
    for name, param in model.named_parameters():
        print(f"Parameter name: {name}")
        print(f"Data type: {param.dtype}")
        print(f"Values: {param.view(-1)[:10]}")  # Print the first 10 values
        print(f"Storage size (bytes): {param.numel() * param.element_size()}")
        print()



# Check quantized weights and biases
# check_quantization(quantized_model)

In [ ]:
output = quantized_model(input_tensor, mask=my_array)

In [ ]:
def main_processing(image):
    resized_image =  resize_nearest_neighbor(image, (224, 224))
    resized_image = bgr_to_rgb(resized_image)
    # gray_image = rgb_to_gray(resized_image)
    resized_image = resized_image.copy()                                        
    image_tensor= transform_image(resized_image, device)
    mask = np.random.randint(1, 2, size=197) 
    
    return image_tensor, mask


In [ ]:
token_counts_per_image = []

In [ ]:
def save_percentage(file_path, average_percent1, average_percent2, average_percent3 ,average_percent4, average, count):
    with open(file_path, 'a') as file:
        file.write(f"Epoch: Average percentage_patch_drop: { average_percent1}, { average_percent2}, { average_percent3}, { average_percent4}, average: {average}, count{count}\n")

In [ ]:
def convert_to_canvas(img_folder, model, device):
    correct_predictions = 0
    total_images = 0
    processed_images = 0
    
    for subfolder_name in os.listdir(img_folder):
        subfolder_path = os.path.join(img_folder, subfolder_name)
        if os.path.isdir(subfolder_path):
            # Process each image in the subfolder
            for img_name in os.listdir(subfolder_path):
                img_path = os.path.join(subfolder_path, img_name)
                if os.path.isfile(img_path) and img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    # Load the image
                    image = cv2.imread(img_path)
                    if image is None:
                        print(f"Failed to load image: {img_path}")
                        continue

                    image_tensor, fast_mask = main_processing(image)
                    
                    # outputs = custom_vit( image_tensor, mask= fast_mask)
                    outputs= quantized_model(image_tensor, mask= fast_mask)
                    predicted_idx = torch.argmax(outputs, dim=1).item()
                    class_name = Path(img_path).parent.name
                    
                    prediction_result = get_key_by_value(predicted_idx, class_name)
                    
                    if prediction_result is True:
                        correct_predictions += 1    
                    processed_images += 1
                    
                    # Print accuracy after processing every 50 images
                    if processed_images % 10 == 0:
                        # yes= ( sum(token_counts_per_image)/ len(token_counts_per_image))
                        # print(len(token_counts_per_image))
                        # print(yes)
                        # w= ( sum(scan)/ len(scan))
                        accuracy_so_far = (correct_predictions / processed_images) * 100
                        # average_percent= 15_15_15_15
                        w=0
                        # average_percent = sum(patch_drop_percentage) / len(patch_drop_percentage)
                        # # print("average percentage embedding_drop " , average_percent , len(patch_drop_percentage))
                        print(f"Accuracy after processing {processed_images} images: {accuracy_so_far:.2f}%")
                        save_average_percentage(txt_path, attention_pruning, accuracy_so_far, processed_images, w )
                    
                    
                    


In [ ]:
# device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
# Example usage
print(device)

In [ ]:
convert_to_canvas(img_folder, pretrained_vit, device)

[Historical saved output; not rerun during repository preparation]
Accuracy after processing 9900 images: 81.42%
Accuracy after processing 9910 images: 81.44%
Accuracy after processing 9920 images: 81.45%
Accuracy after processing 9930 images: 81.47%
Accuracy after processing 9940 images: 81.49%
Accuracy after processing 9950 images: 81.51%
Accuracy after processing 9960 images: 81.53%
Accuracy after processing 9970 images: 81.45%
Accuracy after processing 9980 images: 81.46%
Accuracy after processing 9990 images: 81.48%
Accuracy after processing 10000 images: 81.50%
